##### Library imports

In [ ]:
import re, os, sys
import pandas as pd
import cv2 as cv
import SimpleITK as sitk
import numpy as np 
import json
from skimage import measure,filters
import matplotlib.pyplot as plt

##### Function definitions

In [ ]:
def window_stack_sitk(input_im, window_center=40, window_width=80):
    """
    Clamps values outside [min, max] to the edge values.
    
    Parameters
    ----------
    input_im : sitk.Image
        Input, unwindowed, brain image
    window_center : int, optional
        The center of the Hounsfield Unit (HU) scale in the windowed image. Default is 40 HU (brain).
    window_width : int, optional
        The total HU window width around the window_center, in the windowed image. Default is 80 HU (brain).

    Returns
    -------
    windowed_im : sitk.Image
        Windowed CT image with the desired tissue's visualization optimized (e.g., 0 - 80 HU for brain tissue)
    
    """
    img_min = window_center - (window_width // 2)
    img_max = window_center + (window_width // 2)


    windowed_im = sitk.IntensityWindowing(
        input_im, 
        windowMinimum=float(img_min), 
        windowMaximum=float(img_max), 
        outputMinimum=float(img_min), 
        outputMaximum=float(img_max)
    )
    
    return windowed_im

In [ ]:
def getLargestCC(blobs_labels):
    """Returns the largest connected component from an image containing blobs of discrete labels
    
    Parameters
    ----------
    blobs_labels : numpy.ndarray 
        A collection of discrete blobs (e.g., cerebrospinal fluid [CSF] spaces including the lateral ventricles, longitudinal fissure, sulci etc.)

    Returns
    -------
    largestCC: numpy.ndarray
        The largest connected component from a collection of discrete blobs (e.g. lateral ventricles from a collection of CSF spaces)  
    
    """
    # assume at least 1 CC apart from background
    if blobs_labels.max() == 0:
        raise ValueError('Blank segmentation, inspect processing up to here.')
    #this assumes that the background is the largest CC (label 0)
    largestCC = blobs_labels == np.argmax(np.bincount(blobs_labels.flat)[1:])+1 
    return largestCC

In [ ]:
def rotate_image(image, dimension = 3):
    """Rotates NIfTI axial images so they are upright for visualization. Simple insights tool kit (SITK) utilizes the LPS coordinate system, 
       meaning image voxel coordinates are assumed to increase in the Right --> Left, Anterior --> Posterior, and Inferior --> Superior directions. 
       However, NIfTI images use the RAS coordinate system, which makes axial slices (lateral cross-sections) appear inverted when read with SITK. 
       This function resamples the image such that it is visualized in upright direction, enabling interpretable visualization and processing. 
    
    Parameters
    ----------
    image : sitk.Image
        The original NIfTI image (Note that the direction cosine should be [-1,0,0,0,-1,0,0,0,1])
    dimension : int, optional
        Dimensionality of the original and rotation-corrected image. Default is 3.
    
    Returns
    -------
    rotated_image : sitk.Image
        Rotation-corrected image, with a direction cosine of [1,0,0,0,1,0,0,0,1]
    """

    transform = sitk.AffineTransform(dimension)
    transform.SetCenter(image.TransformContinuousIndexToPhysicalPoint(np.array(image.GetSize())//2.0))
    s = image.GetSpacing()[2]

    matrix = np.array([[1.0,0.0,0.0],[0.0,1.0,0.0],[0.0,0.0,1.0]])

   
    transform.SetMatrix(matrix.ravel())

    extreme_points = [image.TransformIndexToPhysicalPoint((0,0,0)), 
                      image.TransformIndexToPhysicalPoint((image.GetWidth(),0,0)),
                      image.TransformIndexToPhysicalPoint((image.GetWidth(),image.GetHeight(),0)),
                      image.TransformIndexToPhysicalPoint((0,image.GetHeight(),0)),
                      image.TransformIndexToPhysicalPoint((0,0,image.GetDepth())), 
                      image.TransformIndexToPhysicalPoint((image.GetWidth(),0,image.GetDepth())),
                      image.TransformIndexToPhysicalPoint((image.GetWidth(),image.GetHeight(),image.GetDepth())),
                      image.TransformIndexToPhysicalPoint((0,image.GetHeight(),image.GetDepth()))]

    inv_transform = transform.GetInverse()

    extreme_points_transformed = [inv_transform.TransformPoint(pnt) for pnt in extreme_points]
    min_x = min(extreme_points_transformed)[0]
    min_y = min(extreme_points_transformed, key=lambda p: p[1])[1]
    min_z = min(extreme_points_transformed, key=lambda p: p[2])[2]
    max_x = max(extreme_points_transformed)[0]
    max_y = max(extreme_points_transformed, key=lambda p: p[1])[1]
    max_z = max(extreme_points_transformed, key=lambda p: p[2])[2]

    # Use the original spacing (arbitrary decision).
    output_spacing = image.GetSpacing()
    # Identity cosine matrix.   
    output_direction = [1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0]
    # Minimal x,y coordinates are the new origin.
    output_origin = [min_x, min_y, min_z]
    
    # Compute grid size based on the physical size and spacing.
    output_size = image.GetSize()
    
    rotated_image = sitk.Resample(image, output_size, transform, sitk.sitkLinear, output_origin, output_spacing,
                                  output_direction)
    return rotated_image

In [ ]:
def translate_image(image, dimension, ac):
    """
    Translates the physical origin of the image so that the Anterior 
    Commissure (AC) world coordinates are set to (0, 0, 0).
    
    Parameters
    ----------
    image : sitk.Image
        The 3D SimpleITK image to be translated.
    dimension : int
        The dimensionality of the image. (Note: currently unused in the 
        function logic, as the transform is hardcoded to 3D).
    ac : tuple or list of float
        The (x, y, z) world coordinates of the Anterior Commissure.

    Returns
    -------
    translated_image : sitk.Image
        The resampled SimpleITK image with the translated spatial domain.
    """
    transform = sitk.AffineTransform(3)
    transform.SetCenter(image.TransformContinuousIndexToPhysicalPoint(np.array(image.GetSize())//2.0))
    transform.SetTranslation([x for x in ac])
    matrix = np.array([[1.0,0.0,0.0],[0.0,1.0,0.0],[0.0,0.0,1.0]])
    transform.SetMatrix(matrix.ravel())
    extreme_points = [image.TransformIndexToPhysicalPoint((0,0,0)), 
                      image.TransformIndexToPhysicalPoint((image.GetWidth(),0,0)),
                      image.TransformIndexToPhysicalPoint((image.GetWidth(),image.GetHeight(),0)),
                      image.TransformIndexToPhysicalPoint((0,image.GetHeight(),0)),
                      image.TransformIndexToPhysicalPoint((0,0,image.GetDepth())), 
                      image.TransformIndexToPhysicalPoint((image.GetWidth(),0,image.GetDepth())),
                      image.TransformIndexToPhysicalPoint((image.GetWidth(),image.GetHeight(),image.GetDepth())),
                      image.TransformIndexToPhysicalPoint((0,image.GetHeight(),image.GetDepth()))]
    inv_transform = transform.GetInverse()
    extreme_points_transformed = [inv_transform.TransformPoint(pnt) for pnt in extreme_points]
    min_x = min(extreme_points_transformed)[0]
    min_y = min(extreme_points_transformed, key=lambda p: p[1])[1]
    min_z = min(extreme_points_transformed, key=lambda p: p[2])[2]
    max_x = max(extreme_points_transformed)[0]
    max_y = max(extreme_points_transformed, key=lambda p: p[1])[1]
    max_z = max(extreme_points_transformed, key=lambda p: p[2])[2]
    # Use the original spacing (arbitrary decision).
    output_spacing = image.GetSpacing()
    # Identity cosine matrix.   
    output_direction = [1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0]
    # Minimal x,y coordinates are the new origin.
    output_origin = [min_x, min_y, min_z]

    # Compute grid size based on the physical size and spacing.
    output_size = image.GetSize()

    translated_image = sitk.Resample(image, output_size, transform, sitk.sitkLinear,output_origin,output_spacing,
                                      output_direction)
    return translated_image

In [ ]:
def calc_ei_y(axial_data, ac_i, ac_j, ac_k, ei_y_processing_parameters, visualize = False):

    """
    Calculates the modified y-axis Evans Index (EI_y) from axial brain CT data.

    This function iterates through axial slices superior to the anterior commissure (AC)
    to evaluate the anterior-posterior (y-axis) length of the frontal horns. It dynamically 
    adjusts a column threshold to separate the left and right ventricles. The final EI_y 
    is computed as the maximum frontal horn y-length divided by the maximum y-length 
    of the brain mask on the optimal slice.

    Parameters
    ----------
    axial_data : numpy.ndarray
        A 3D array representing the axial CT volume.
    ac_i : float or int
        The x-coordinate (column index) of the anterior commissure.
    ac_j : float or int
        The y-coordinate (row index) of the anterior commissure.
    ac_k : float or int
        The z-coordinate (slice index) of the anterior commissure.
    ei_y_processing_parameters : dict
        Dictionary of predefined parameters for image processing and ROI definitions.
    visualize : bool, optional
        If True, displays intermediate masks and the final calculated lengths overlaid 
        on the optimal slice. Default is False.

    Returns
    -------
    ei_y : float or None
        The calculated y-axis Evans Index. Returns None if a valid measurement 
        cannot be found.
    opt_plane : int or None
        The axial slice index (z-coordinate) where the maximum y-length was found. 
        Returns None if extraction fails.
    """

    c_thresh = ei_y_processing_parameters["c_thresh"]
    dec_thresh = ei_y_processing_parameters["dec_thresh"]
    ei_y_num_slices_above_ac_to_eval = ei_y_processing_parameters["ei_y_num_slices_above_ac_to_eval"]
    axial_gaussian_blur_initial = ei_y_processing_parameters["axial_gaussian_blur_initial"]
    axial_adaptive_thresh_size = ei_y_processing_parameters["axial_adaptive_thresh_size"]
    axial_adaptive_thresh_mean_thresh = ei_y_processing_parameters["axial_adaptive_thresh_mean_thresh"]
    otsu_adaptive_diff_thresh = ei_y_processing_parameters["otsu_adaptive_diff_thresh"]
    otsu_adaptive_horizontal_thresh = ei_y_processing_parameters["otsu_adaptive_horizontal_thresh"]
    otsu_adaptive_vert_thresh = ei_y_processing_parameters["otsu_adaptive_vert_thresh"]
    lv_h_roi_col_thresh_prcnt = ei_y_processing_parameters["lv_h_roi_col_thresh_prcnt"]
    lv_h_roi_row_thresh = ei_y_processing_parameters["lv_h_roi_row_thresh"]
    midline_detected_vert_thresh = ei_y_processing_parameters["midline_detected_vert_thresh"]
    
    y2 = int(round(ac_j))    
    ac_i_int = int(round(ac_i))   

    final_x1_opt = None
    final_y1_opt = None
    full_brain_mask_opt = None
    opt_plane = None
    axial_pl_opt = None

    max_dec_thresh_limit = 20

    while True:
        # Prevent infinite loops if valid ventricle masks are never found
        if dec_thresh > max_dec_thresh_limit:
            print("Warning: Exceeded threshold adjustment limits. Could not calculate EI_y.")
            return None, None
            
        max_fh_length = 0 #resetting value for each image, each thresh

        for plane in range(int(round(ac_k)), int(round(ac_k))+ei_y_num_slices_above_ac_to_eval):
            if plane >= axial_data.shape[0]:
                break
                
            axial_brain_im = axial_data[plane,:,:]
            #Denoising
            axial_gauss_blurred = cv.GaussianBlur(np.uint8(axial_brain_im),(axial_gaussian_blur_initial,axial_gaussian_blur_initial),
                                                  cv.BORDER_DEFAULT)  
            thresh = filters.threshold_otsu(axial_gauss_blurred)
            axial_bin = axial_gauss_blurred > thresh
            
            axial_ad_th = cv.adaptiveThreshold(axial_gauss_blurred, 1, cv.ADAPTIVE_THRESH_MEAN_C,
                                                cv.THRESH_BINARY, axial_adaptive_thresh_size, axial_adaptive_thresh_mean_thresh) 

            #The background needs to be subtracted from the adaptive threshold because background pixels are classified as foreground 
            #because of local thresholding
            blobs_labels = measure.label(axial_brain_im, background = 0)
            background_pixels = blobs_labels == np.argmax(np.bincount(blobs_labels.flatten()))

            axial_brain_mask = np.where(background_pixels>0,0, axial_ad_th)

            #Floodfilling is used to extract the entire brain mask 
            im_th = axial_brain_mask.copy()
            h, w = im_th.shape[:2]
            mask = np.zeros((h+2, w+2), np.uint8)
            im_floodfill = im_th.copy()
            # Floodfill from point (0, 0)
            res = cv.floodFill(np.uint8(im_floodfill), mask, (0,0), 255)
            full_brain_mask = (1-res[2]).copy()
            full_brain_mask = full_brain_mask[1:h+1,1:w+1]
            full_brain_mask = np.where(getLargestCC(measure.label(full_brain_mask, background = 0))>0,1,0)
            [fbr, fbc] = full_brain_mask.nonzero()

            M = cv.moments(np.uint8(full_brain_mask))

            if M["m00"] == 0:
                continue
                
            cX = int(M["m10"] / M["m00"])
            cY = int(M["m01"] / M["m00"])
            #This gives a primary segmentation 
            adaptive_ventricles = np.where(full_brain_mask - axial_brain_mask>0,1,0)

            if np.sum(adaptive_ventricles)==0:
                continue

            tm = np.zeros(adaptive_ventricles.shape)
            tm[:adaptive_ventricles.shape[0]//2,:]=1

            ventricles = adaptive_ventricles

            try:
                otsu_ventricles = getLargestCC(measure.label(np.where(full_brain_mask - axial_bin>0,1,0),background = 0))
                if (np.sum(otsu_ventricles*tm) > 0) and (np.sum((adaptive_ventricles)*tm) > 0):
                    diff_otsu_adaptive = np.where(otsu_ventricles-adaptive_ventricles>0,1,0)*tm
                    if (np.sum(diff_otsu_adaptive) > 0):
                        
                        Ma = cv.moments(np.uint8(adaptive_ventricles)*tm)
                        aX = int(Ma["m10"] / Ma["m00"])
                        aY = int(Ma["m01"] / Ma["m00"])
                        diff_density = np.sum(diff_otsu_adaptive)/np.sum(otsu_ventricles*tm)
                        Md = cv.moments(np.uint8(diff_otsu_adaptive))
                        dX = int(Md["m10"] / Md["m00"])
                        dY = int(Md["m01"] / Md["m00"])

                        diff_check = ((diff_density > otsu_adaptive_diff_thresh) and (np.abs(aX-dX) < otsu_adaptive_horizontal_thresh) and
                                     (np.abs(aY-dY) < otsu_adaptive_vert_thresh)) 

                        if diff_check:
                            # print('OTSU picked up')
                            ventricles = np.logical_or(adaptive_ventricles, diff_otsu_adaptive)
            except (ValueError, IndexError, ZeroDivisionError):
                ventricles = adaptive_ventricles


            #This ensures that that the left and right parts of the FH remain connected 
            h_roi = np.zeros(ventricles.shape)
            [r_v, c_v] = ventricles.nonzero()
            mr_v, mc_v = (max(r_v) + min(r_v))//2, (max(c_v) + min(c_v))//2
            [fr, fc] = full_brain_mask.nonzero()
            fbc_range = max(fc) - min(fc)
            col_thresh = int(fbc_range*lv_h_roi_col_thresh_prcnt)
           
            #This step extracts any remaining background pixels and eliminates them 
            # Prevent negative indexing
            r_start = max(0, mr_v - lv_h_roi_row_thresh)
            c_start = max(0, cX - col_thresh)
            c_end = min(ventricles.shape[1], cX + col_thresh)
            h_roi[r_start:mr_v, c_start:c_end] = 1
            
            ventricles_connected = np.logical_or(h_roi, ventricles)
            try:
                [r,c] = (1-getLargestCC(measure.label(ventricles_connected, background = 0))).nonzero()
            except (ValueError, IndexError): 
                continue
                
            ventricles_connected_1 = ventricles.copy()
            ventricles_connected_1[r,c] = 0
            #This gets rid of the lower part of the lateral ventricles
            fh_roi = np.ones(ventricles_connected_1.shape)
            fh_roi[y2:,:]=0

            ventricles_fh = ventricles_connected_1*fh_roi

            if np.sum(ventricles_fh)==0:
                continue

            [vfh_r,vfh_c] = ventricles_fh.nonzero()
            if (max(vfh_r) - min(vfh_r)) > midline_detected_vert_thresh*(max(vfh_c) - min(vfh_c)):
                continue

            slice_idx = max(0, ac_i_int - (c_thresh - dec_thresh))
            left_ven = ventricles_fh[:, :slice_idx]
            
            slice_idx_right = min(ventricles_fh.shape[1], ac_i_int + (c_thresh - dec_thresh))
            right_ven = ventricles_fh[:, slice_idx_right:]

            
            try:
                left_ven = getLargestCC(measure.label(left_ven, background = 0))
            except (AssertionError, ValueError, IndexError):
                pass
            try:
                right_ven = getLargestCC(measure.label(right_ven, background = 0))
            except (AssertionError, ValueError, IndexError):
                pass
            
            if visualize:
                plt.imshow(axial_gauss_blurred, cmap='gray')
                plt.title('Gauss blurred')
                plt.show()
                plt.imshow(ventricles_connected_1, cmap='gray')
                plt.title('OR between ventricles and middle only closing')
                plt.show()
                plt.imshow(left_ven, cmap='gray')
                plt.title('Left ventricles')
                plt.show()
                plt.imshow(right_ven, cmap='gray')
                plt.title('Right ventricles')
                plt.show()

            [lr,lc] = left_ven.nonzero()
            [rr, rc] = right_ven.nonzero()
        

            if not(len(lr) > 0 and len(lc) > 0 and len(rr) > 0 and len(rc) > 0):
                continue 

            cur_max_l_fh_len = 0

            start_lc = min(lc)
            stop_lc = max(lc) 
            for c in range(start_lc,stop_lc):
                try:
                    fh_len = y2 - min(lr[lc==c])
                except ValueError:
                    continue
                if fh_len > cur_max_l_fh_len:
                    cur_max_l_fh_len = fh_len
                    ly1 = min(lr[lc==c])
                    lx1 = int(np.round(np.mean(lc[lr == ly1])))

            cur_max_r_fh_len = 0

            start_rc = min(rc)
            stop_rc = max(rc) 
            for c in range(start_rc,stop_rc):
                try:
                    fh_len = y2 - min(rr[rc==c])
                except ValueError:
                    continue
                if fh_len > cur_max_r_fh_len:
                    cur_max_r_fh_len = fh_len
                    ry1 = min(rr[rc==c])
                    rx1 = int(np.round(np.mean(rc[rr == ry1])))


            if (cur_max_l_fh_len == 0) and (cur_max_r_fh_len == 0):
                continue

            fh_len_pl = 0   #resetting value for each plane
            if cur_max_l_fh_len >= cur_max_r_fh_len:
                fh_len_pl = y2 - ly1 
                y1_pl = ly1
                x1_pl = lx1
            else:
                fh_len_pl = y2 - ry1
                y1_pl = ry1
                x1_pl = rx1 + ac_i_int + c_thresh - dec_thresh

            if fh_len_pl > max_fh_length: #updating opt vals after each plane's exam
                max_fh_length = fh_len_pl
                final_x1_opt, final_y1_opt = x1_pl, y1_pl
                full_brain_mask_opt = full_brain_mask
                opt_ven_image = ventricles_fh
                opt_left_ven = left_ven
                opt_right_ven = right_ven
                opt_plane = plane
                axial_pl_opt = axial_brain_im

        if max_fh_length == 0: #if after all planes are evaluated at a thresh and it is still 0, relaxing thresh 
            dec_thresh = dec_thresh + 1
        else: #if not, produce output
            [fbr, fbc] = full_brain_mask_opt.nonzero()

            max_y_len = 0
            for c in sorted(np.unique(fbc)):
                y_len = max(fbr[fbc==c]) - min(fbr[fbc==c])
                if y_len > max_y_len:
                    max_y_len = y_len 
                    fb_x = c
                    fb_y2 = max(fbr[fbc==c])
                    fb_y1 = min(fbr[fbc==c])

            ei_y = max_fh_length/max_y_len

            fig, ax = plt.subplots(figsize = (10,10))
            ax.imshow(axial_pl_opt,cmap = 'gray')

            #offset to make the traced lengths clear if their x coordinates are too close
            if abs(final_x1_opt - fb_x) < 4:
                final_x1_opt = final_x1_opt - 4
                
            [ax.scatter(x,y,marker = 'x',color = 'w',s = 2) for x,y in zip([final_x1_opt]*(y2-final_y1_opt),
                                                                           [y for y in range(final_y1_opt,y2)])]
            [ax.scatter(x,y,marker = 'x',color = 'w',s = 2) for x,y in zip([fb_x]*(fb_y2-fb_y1),
                                                                           [y for y in range(fb_y1,fb_y2)])]

            plt.show()

            return ei_y, opt_plane

##### Data setup

###### Before running the pipeline, make a copy of config.template.json, rename it to config.json, and update the paths to point to your local dataset.

In [ ]:
# Load the configuration file
with open('config.json', 'r') as file:
    config = json.load(file)

# Assign variables dynamically
data_path = config['data_path']
axial_acpc_vol_aligned_name = config['axial_acpc_vol_aligned_name']
info_csv_path = config['info_csv_path_w_name']
scan_id_col_name = config['scan_id_col_name']
pat_id_col_name = config['pat_id_col_name']
ac_coordinates_col_name = config['ac_coordinates_col_name']
pc_coordinates_col_name = config['pc_coordinates_col_name']

output_folder = config['output_csv_write_folder']

print(f"Loading data from: {data_path}")

In [ ]:
info_df = pd.read_csv(info_csv_path)
info_df[scan_id_col_name] = info_df[scan_id_col_name].str.strip("'")

In [ ]:
accnum_list = info_df[scan_id_col_name].values

In [ ]:
len(accnum_list), len(info_df)

In [ ]:
#dataframe which will contain track of the computed EI-y values for each scan
ei_y_df = pd.DataFrame()

###### Predefined, emperically determined processing parameters corresponding to ROI sizes, positions, and global/local thresholding parameters. 
The following parameters work for a diverse cohort of chronic neurodegenerative conditions spanning Normal Pressure Hydrocephalus, Alzheimer's disease, post-traumatic encephalomalacia, and headache. So they are recommended to be used directly for patient scans without acute pathology like significant midline shifts, bleeds, or mass effects. 

In [ ]:
ei_y_processing_parameters = dict()
ei_y_processing_parameters["c_thresh"] = 7
ei_y_processing_parameters["dec_thresh"] = 0
ei_y_processing_parameters["ei_y_num_slices_above_ac_to_eval"] = 20
ei_y_processing_parameters["axial_gaussian_blur_initial"] = 3
ei_y_processing_parameters["axial_adaptive_thresh_size"] = 85
ei_y_processing_parameters["axial_adaptive_thresh_mean_thresh"] = 7.5
ei_y_processing_parameters["otsu_adaptive_diff_thresh"] = 0.35
ei_y_processing_parameters["otsu_adaptive_horizontal_thresh"] = 7
ei_y_processing_parameters["otsu_adaptive_vert_thresh"] = 40
ei_y_processing_parameters["lv_h_roi_col_thresh_prcnt"] = 0.15
ei_y_processing_parameters["lv_h_roi_row_thresh"] = 20
ei_y_processing_parameters["midline_detected_vert_thresh"] = 1.5

##### EI-y pipeline

In [ ]:
for acc_num in accnum_list:
    
    pat = info_df[pat_id_col_name][info_df[scan_id_col_name]==acc_num].values[0]

    #Read in AC coordinates
    aca_str = info_df[ac_coordinates_col_name][(info_df[scan_id_col_name]==acc_num)].values[0]
    [x,y,z] = aca_str.split(",")
    aca_x,aca_y,aca_z = [float(x.strip("[")),float(y),float(z.strip("]"))]            
   
    print(f'Patient is {pat} and acc_num is {acc_num}')

    #Read in AC-PC aligned axial volume
    study_path = os.path.join(data_path, acc_num, axial_acpc_vol_aligned_name)
  
    axial_img = rotate_image(sitk.ReadImage(study_path))
    ax_img_brain = window_stack_sitk(axial_img)
    ac = [aca_x, aca_y, aca_z]
    ax_img_brain_translated = translate_image(ax_img_brain, 3, ac)
    axial_data = sitk.GetArrayFromImage(ax_img_brain_translated)
    N = axial_data.shape        
    ac_i, ac_j, ac_k = ax_img_brain_translated.TransformPhysicalPointToContinuousIndex([0,0,0])

    #Compute EI-y    
    ei_y, opt_plane = calc_ei_y(axial_data, ac_i, ac_j, ac_k, ei_y_processing_parameters)
    
    ei_y_df = pd.concat([ei_y_df, pd.DataFrame({pat_id_col_name:[pat],scan_id_col_name:["'" + acc_num + "'"],'ei_y':[ei_y]})], 
                         ignore_index = True)
    
        
        

In [ ]:
len(ei_y_df), sum(ei_y_df['ei_y'].isna())

In [ ]:
#Examine histogram of computed values
plt.hist(ei_y_df['ei_y'])
plt.xticks([x for x in np.arange(0, 0.5, 0.1)])
plt.show()

In [ ]:
#Check for any extreme values and inspect the corresponding outputs above for computational errors
min_ei_y_lim = 0.1
max_ei_y_lim = 0.5

ei_y_df[(ei_y_df['ei_y'] <= min_ei_y_lim) | (ei_y_df['ei_y'] >= max_ei_y_lim)]

In [ ]:
ei_y_df.to_csv(os.path.join(output_folder, 'ei_y.csv'))